In [434]:
import numpy as np
import pandas as pd
from casadi import *
from scipy.spatial import KDTree
from scipy.interpolate import interp1d
import plotly.graph_objects as go
from Functions.Utils import *
from Functions.Graphs import *
from scipy.interpolate import CubicSpline
import mosek


1. LOAD RACETRACK DATA & SPATIAL PARAMETRIZATION

In [435]:
size = 1
path = r'DyntheticDataset\RaceTrack4.csv'
df = pd.read_csv(path)
x_mid = df['x_coords'].values[:]*size
y_mid = df['y_coords'].values[:]*size
#x_mid,y_mid = interpolar_racetrack(x_mid,y_mid)

# Ensure the track loops closed cleanly
if not np.allclose([x_mid[0], y_mid[0]], [x_mid[-1], y_mid[-1]]):
    x_mid = np.append(x_mid, x_mid[0])
    y_mid = np.append(y_mid, y_mid[0])

track_points = np.vstack((x_mid, y_mid)).T
track_tree = KDTree(track_points)

# Calculate cumulative distance (arc length 's') along track coordinates
dx = np.diff(x_mid)
dy = np.diff(y_mid)
segment_lengths = np.sqrt(dx**2 + dy**2)
s_coor = np.insert(np.cumsum(segment_lengths), 0, 0.0)
track_length = s_coor[-1]

# Linear interpolators to find exact continuous (x, y) coordinates for any distance s
get_x_at_s = interp1d(s_coor, x_mid, kind='linear', bounds_error=False, fill_value="extrapolate")
get_y_at_s = interp1d(s_coor, y_mid, kind='linear', bounds_error=False, fill_value="extrapolate")

2. CASADI OPTI PROBLEM CONFIGURATION (Defined ONCE outside loop)


In [436]:
dt = 0.0125
sim_steps = 1000
V_cruising = 17
V_max = 25
V_turning = 10
n_horizon = 25
l = 2.5 # Wheelbase

opti = Opti()

# Decision variables over the prediction horizon
X = opti.variable(4, n_horizon + 1) # States: [x_pos, y_pos, v, psi]
U = opti.variable(2, n_horizon)     # Controls: [a, delta]

# Parameters
X0 = opti.parameter(4)
X_REF = opti.parameter(n_horizon)
Y_REF = opti.parameter(n_horizon)
V_REF = opti.parameter(n_horizon)

obj = 0
opti.subject_to(X[:, 0] == X0)

# Build the execution graph symbolic tree once
for k in range(n_horizon):
    x_k   = X[0, k]
    y_k   = X[1, k]
    v_k   = X[2, k]
    psi_k = X[3, k]
    
    a_k     = U[0, k]
    delta_k = U[1, k]

    beta = atan(0.5 * tan(delta_k))
    
    next_x   = x_k + dt * (v_k * cos(psi_k + beta))
    next_y   = y_k + dt * (v_k * sin(psi_k + beta))
    next_v   = v_k + dt * a_k
    next_psi = psi_k + dt * ((v_k / l) * sin(beta))

    opti.subject_to(X[0, k+1] == next_x)
    opti.subject_to(X[1, k+1] == next_y)
    opti.subject_to(X[2, k+1] == next_v)
    opti.subject_to(X[3, k+1] == next_psi)

    # Stage cost
    obj += (x_k - X_REF[k])**2 + (y_k - Y_REF[k])**2 + 5 * (v_k - V_REF[k])**2
    obj += 1.5 * a_k**2 + 0.25 * delta_k**2

    if k > 0:
        obj += 5.0 * (U[0, k] - U[0, k-1])**2  # Jerk penalty (smooths out acceleration)
        obj += 1.0 * (U[1, k] - U[1, k-1])**2

# CRITICAL FIX: Replaced [-1] negative indexing with explicit [n_horizon - 1]
obj += (X[0, n_horizon] - X_REF[n_horizon - 1])**2 + \
       (X[1, n_horizon] - Y_REF[n_horizon - 1])**2 + \
       5.0 * (X[2, n_horizon] - V_REF[n_horizon - 1])**2

opti.minimize(obj)

# Bounds
opti.subject_to(opti.bounded(-7.0, U[0, :], 4))                 
opti.subject_to(opti.bounded(-np.deg2rad(30), U[1, :], np.deg2rad(30))) 
opti.subject_to(opti.bounded(0.0, X[2, :], V_cruising*1.25))                  

# Setup solver configuration options once
opts = {"ipopt.print_level": 0, "print_time": 0, "ipopt.warm_start_init_point": "yes"}
opti.solver('ipopt', opts)


3. CLOSED-LOOP SIMULATION LOOP


In [437]:
s_total_traveled = 0.0
last_current_idx = 0

x0_start = np.array([50, y_mid[0], 0.0, 0.0])
x_current = x0_start.copy()
current_time = 0.0

t_history = [current_time]
x_history = [x_current[0]]
y_history = [x_current[1]]
v_history = [x_current[2]]
v_ref_history = []
v_k_history = []
v_horizon = []
acc_history = [0]
delta_history = [0]
psi_history = [0]
is_turning_sharp = False
turning = []

# Continuous trajectory memory for warm-starting
last_sol_X = np.tile(x_current.reshape(4, 1), (1, n_horizon + 1))
last_sol_U = np.zeros((2, n_horizon))

for step in range(sim_steps):
    if step % 250 == 0 :print('step:',step)
    _, current_idx = track_tree.query([x_current[0], x_current[1]])
    
    idx_diff = current_idx - last_current_idx
    if idx_diff < -len(x_mid)/2: 
        idx_diff += len(x_mid)
    elif idx_diff > len(x_mid)/2:
        idx_diff -= len(x_mid)
        
    if idx_diff > 0:
        s_total_traveled += np.sum(segment_lengths[last_current_idx:current_idx])
    last_current_idx = current_idx

    # Generate Time-Varying Horizon Reference Profiles
    s_projected = s_coor[current_idx]
    x_ref_horizon = np.zeros(n_horizon)
    y_ref_horizon = np.zeros(n_horizon)
    v_ref_horizon = np.zeros(n_horizon)
    #is_turning_sharp = (x_current[3] >= np.deg2rad(10)) or (x_current[3] <= -np.deg2rad(10))
    #is_turning_sharp = (abs(x_current[3]) >= np.deg2rad(5))
    #turning.append(is_turning_sharp)

    for k in range(n_horizon):
        s_projected += max(x_current[2], 1.5) * dt 
        s_wrapped = s_projected % track_length
        
        x_ref_horizon[k] = get_x_at_s(s_wrapped)
        y_ref_horizon[k] = get_y_at_s(s_wrapped)
        
        if s_total_traveled >= track_length * 1.0:
            
            v_ref_horizon[k] = 0.0

        elif is_turning_sharp:
            #if x_current[2] < V_turning*0.9:
            #    v_ref_horizon[k] = V_turning*1.05
            #elif x_current[2] >= V_turning:
            #    v_ref_horizon[k] = V_turning*0.8
            v_ref_horizon[k] = V_turning
            
        else:
            #if x_current[2] <= V_max:
            #    v_ref_horizon[k] = V_cruising*1.1
            #elif x_current[2] > V_cruising:
            #    v_ref_horizon[k] = V_cruising*0.8
            v_ref_horizon[k] = V_cruising
    # Update parameters
    opti.set_value(X0, x_current)
    opti.set_value(X_REF, x_ref_horizon)
    opti.set_value(Y_REF, y_ref_horizon)
    opti.set_value(V_REF, v_ref_horizon)

    # Seed guesses
    opti.set_initial(X, last_sol_X)
    opti.set_initial(U, last_sol_U)

    try:
        sol = opti.solve()
        u_control = sol.value(U[:, 0])
        last_sol_X = sol.value(X)
        last_sol_U = sol.value(U)
    except Exception as e:
        # Fallback: Retrieve the last non-optimal/sub-optimal values instead of crashing instantly
        u_control = opti.debug.value(U[:, 0])
        last_sol_X = opti.debug.value(X)
        last_sol_U = opti.debug.value(U)

    # --- Plant Simulator (Process Step using Forward Euler) ---
    beta_sim = np.arctan(0.5 * np.tan(u_control[1]))
    x_next = x_current[0] + dt * (x_current[2] * np.cos(x_current[3] + beta_sim))
    y_next = x_current[1] + dt * (x_current[2] * np.sin(x_current[3] + beta_sim))
    v_next = x_current[2] + dt * u_control[0]
    psi_next = x_current[3] + dt * ((x_current[2] / l) * np.sin(beta_sim))

    #is_turning_sharp = (abs(u_control[1]) >= np.deg2rad(15))
    is_turning_sharp = (u_control[1] >= np.deg2rad(8)) or (u_control[1] <= -np.deg2rad(8))
    turning.append(is_turning_sharp)
    
    x_current = np.array([x_next, y_next, v_next, psi_next])
    current_time += dt

    t_history.append(current_time)
    x_history.append(x_current[0])
    y_history.append(x_current[1])
    v_history.append(x_current[2])
    v_k_history.append(last_sol_X[2])
    v_ref_history.append(v_ref_horizon[0])
    acc_history.append(u_control[0])
    psi_history.append(x_current[3])
    delta_history.append(u_control[1])
    v_horizon.append(v_ref_horizon)

    if s_total_traveled >= track_length and x_current[2] < 0.05:
        print(f"Simulation completed! Lap finished and vehicle stopped on centerline at step {step}.")
        break

# Convert historical data into numpy arrays for rendering 

t_history = np.array(t_history)
x_history = np.array(x_history)
y_history = np.array(y_history)
v_history = np.array(v_history)
v_ref_history = np.array(v_ref_history)
acc_history = np.array(acc_history)
delta_history = np.array(delta_history)
delta_history = np.rad2deg(delta_history)

psi_history = np.array(psi_history)
psi_history = np.rad2deg(psi_history)
turning = [0 if t==False else 1 for t in turning]
turning = np.array(turning)


step: 0


CasADi - 2026-06-23 13:41:07 WARNING("KeyboardInterruptException") [.../casadi/interfaces/ipopt/ipopt_nlp.cpp:196]


step: 250
step: 500
step: 750


In [ ]:
PlotSeriesPLY(xSeries=[t_history,t_history],ySeries=[v_ref_history,v_history],w=800,h=300)
PlotSeriesPLY(xSeries=[t_history],ySeries=[acc_history],w=800,h=300)

In [ ]:
print(V_max)
print(V_cruising*1)

25
20


4. PLOT RESULTS

In [ ]:
data = [x_mid, y_mid, x_history,y_history, t_history, delta_history, psi_history, v_history]
PlotMPCTracksPLY(data,width=800,height=500)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_mid, y=y_mid, 
    mode='lines', 
    name='Track Midline Reference', 
    line=dict(color='lightgray', width=2, dash='dash')
))

fig.add_trace(go.Scatter(
    x=x_history, 
    y=y_history, 
    mode='lines+markers', 
    name='Vehicle Profile Path',
    line=dict(color='rgba(100,100,100,0.3)', width=2),
    marker=dict(
        size=5,
        color=turning,
        colorscale='Jet',
        cmin=np.min(turning),
        cmax=np.max(turning),
        colorbar=dict(
            title=dict(text="Speed (m/s)", side="bottom"),
            ticks="outside"
        ),
        showscale=True
    ),
    text=[f"Speed: {v:.2f} m/s" for v in v_history],
    hoverinfo='text+x+y'
))

fig.update_layout(
    title="Discrete MPC: Speed Profile Heatmap Across Lap Mission",
    xaxis_title="X Coordinate (meters)", 
    yaxis_title="Y Coordinate (meters)",
    yaxis=dict(scaleanchor="x", scaleratio=1), 
    showlegend=True,
    template="plotly_white"
)

fig.show()

In [451]:
# %%
import numpy as np
import pandas as pd
from casadi import *
from scipy.spatial import KDTree
from scipy.interpolate import interp1d
import plotly.graph_objects as go
from Functions.Utils import *
from Functions.Graphs import *
from scipy.interpolate import CubicSpline

# %% [markdown]
# 1. LOAD RACETRACK DATA & SPATIAL PARAMETRIZATION

# %%
size = 1
path = r'DyntheticDataset\RaceTrack4.csv'
df = pd.read_csv(path)
x_mid = df['x_coords'].values[:] 
y_mid = df['y_coords'].values[:] 

# Ensure the track loops closed cleanly
if not np.allclose([x_mid[0], y_mid[0]], [x_mid[-1], y_mid[-1]]):
    x_mid = np.append(x_mid, x_mid[0])
    y_mid = np.append(y_mid, y_mid[0])

track_points = np.vstack((x_mid, y_mid)).T
track_tree = KDTree(track_points)

# Calculate cumulative distance (arc length 's') along track coordinates
dx = np.diff(x_mid)
dy = np.diff(y_mid)
segment_lengths = np.sqrt(dx**2 + dy**2)
s_coor = np.insert(np.cumsum(segment_lengths), 0, 0.0)
track_length = s_coor[-1]

# Linear interpolators to find exact continuous (x, y) coordinates for any distance s
get_x_at_s = interp1d(s_coor, x_mid, kind='linear', bounds_error=False, fill_value="extrapolate")
get_y_at_s = interp1d(s_coor, y_mid, kind='linear', bounds_error=False, fill_value="extrapolate")

# %% [markdown]
# 2. CASADI OPTI PROBLEM CONFIGURATION (Defined ONCE outside loop)

# %%
dt = 0.0125
sim_steps = 3000
V_cruising = 16
V_max = 25
V_turning = 10
n_horizon = 25
track_percentual = 0.9
l = 2.5 # Wheelbase

opti = Opti()

# Decision variables over the prediction horizon
X = opti.variable(4, n_horizon + 1) # States: [x_pos, y_pos, v, psi]
U = opti.variable(2, n_horizon)     # Controls: [a, delta]

# Parameters
X0 = opti.parameter(4)
X_REF = opti.parameter(n_horizon)
Y_REF = opti.parameter(n_horizon)
V_REF = opti.parameter(n_horizon)
U_PREV = opti.parameter(2)          # Last physically executed control action parameter [a_prev, delta_prev]

obj = 0
opti.subject_to(X[:, 0] == X0)

# Build the execution graph symbolic tree once
for k in range(n_horizon):
    x_k   = X[0, k]
    y_k   = X[1, k]
    v_k   = X[2, k]
    psi_k = X[3, k]
    
    a_k     = U[0, k]
    delta_k = U[1, k]

    beta = atan(0.5 * tan(delta_k))
    
    next_x   = x_k + dt * (v_k * cos(psi_k + beta))
    next_y   = y_k + dt * (v_k * sin(psi_k + beta))
    next_v   = v_k + dt * a_k
    next_psi = psi_k + dt * ((v_k / l) * sin(beta))

    opti.subject_to(X[0, k+1] == next_x)
    opti.subject_to(X[1, k+1] == next_y)
    opti.subject_to(X[2, k+1] == next_v)
    opti.subject_to(X[3, k+1] == next_psi)

    # Base state and input magnitudes stage cost
    obj += (x_k - X_REF[k])**2 + (y_k - Y_REF[k])**2 + 5 * (v_k - V_REF[k])**2
    obj += 1.5 * a_k**2 + 0.25 * delta_k**2

    # Slew rate penalties inside the preview horizon steps
    if k > 0:
        obj += 15.0 * (U[0, k] - U[0, k-1])**2  # Jerk penalty
        obj += 2.0 * (U[1, k] - U[1, k-1])**2   # Steering rate penalty

# Smooth transitions linking external past inputs directly to the initial horizon decision step
obj += 25.0 * (U[0, 0] - U_PREV[0])**2
obj += 2.0 * (U[1, 0] - U_PREV[1])**2

# Terminal cost step calculation
obj += (X[0, n_horizon] - X_REF[n_horizon - 1])**2 + \
       (X[1, n_horizon] - Y_REF[n_horizon - 1])**2 + \
       5.0 * (X[2, n_horizon] - V_REF[n_horizon - 1])**2

opti.minimize(obj)

# Operational Bound limits
opti.subject_to(opti.bounded(-5.0, U[0, :], 3))                 
opti.subject_to(opti.bounded(-np.deg2rad(30), U[1, :], np.deg2rad(30))) 
opti.subject_to(opti.bounded(0.0, X[2, :], V_cruising*1))                  

# Setup solver configuration options once
opts = {"ipopt.print_level": 0, "print_time": 0, "ipopt.warm_start_init_point": "yes"}
opti.solver('ipopt', opts)


# %% [markdown]
# 3. CLOSED-LOOP SIMULATION LOOP

# %%
s_total_traveled = 0.0
last_current_idx = 0

x0_start = np.array([50, y_mid[0], 0.0, 0.0])
x_current = x0_start.copy()
current_time = 0.0

# Memory array tracking previous system inputs (Starts at 0.0 standstill condition)
u_executed_prev = np.array([0.0, 0.0])

t_history = [current_time]
x_history = [x_current[0]]
y_history = [x_current[1]]
v_history = [x_current[2]]
v_ref_history = []
v_k_history = []
v_horizon = []
acc_history = [0]
delta_history = [0]
psi_history = [0]
is_turning_sharp = False
turning = []

# Continuous trajectory memory for warm-starting
last_sol_X = np.tile(x_current.reshape(4, 1), (1, n_horizon + 1))
last_sol_U = np.zeros((2, n_horizon))

for step in range(sim_steps):
    if step % 250 == 0: print('step:', step)
    _, current_idx = track_tree.query([x_current[0], x_current[1]])
    
    idx_diff = current_idx - last_current_idx
    if idx_diff < -len(x_mid)/2: 
        idx_diff += len(x_mid)
    elif idx_diff > len(x_mid)/2:
        idx_diff -= len(x_mid)
        
    if idx_diff > 0:
        s_total_traveled += np.sum(segment_lengths[last_current_idx:current_idx])
    last_current_idx = current_idx

    # Generate Time-Varying Horizon Reference Profiles
    s_projected = s_coor[current_idx]
    x_ref_horizon = np.zeros(n_horizon)
    y_ref_horizon = np.zeros(n_horizon)
    v_ref_horizon = np.zeros(n_horizon)

    for k in range(n_horizon):
        s_projected += max(x_current[2], 1.5) * dt 
        s_wrapped = s_projected % track_length
        
        x_ref_horizon[k] = get_x_at_s(s_wrapped)
        y_ref_horizon[k] = get_y_at_s(s_wrapped)
        
        if s_total_traveled >= track_length * track_percentual:
            v_ref_horizon[k] = 0.0

        elif is_turning_sharp:
            if ((u_control[1]) > np.deg2rad(3)) and (u_control[1] < np.deg2rad(6)):
                V_turning2 = V_turning+3
            elif ((u_control[1]) >= np.deg2rad(6)) and (u_control[1] < np.deg2rad(9)):
                V_turning2 = V_turning+2
            elif ((u_control[1]) >= np.deg2rad(9)) and (u_control[1] < np.deg2rad(12)):
                V_turning2 = V_turning
            elif ((u_control[1]) >= np.deg2rad(12)):
                V_turning2 = V_turning-2

            if ((u_control[1]) < np.deg2rad(-3)) and (u_control[1] > np.deg2rad(-6)):
                V_turning2 = V_turning+3
            elif ((u_control[1]) <= np.deg2rad(-6)) and (u_control[1] > np.deg2rad(-9)):
                V_turning2 = V_turning+2
            elif ((u_control[1]) <= np.deg2rad(-9)) and (u_control[1] > np.deg2rad(-12)):
                V_turning2 = V_turning
            elif ((u_control[1]) <= np.deg2rad(-12)):
                V_turning2 = V_turning-2


            if x_current[2] > V_turning2:
                v_ref_horizon[k] = V_turning2*0.9
            elif x_current[2] <= V_turning2:
                v_ref_horizon[k] = V_turning2*1.1
            #elif x_current[2] >= V_turning2*1.1:
            #    v_ref_horizon[k] = V_turning2*0.8

        else:
            v_ref_horizon[k] = V_cruising

    # Update parameters mapping arrays
    opti.set_value(X0, x_current)
    opti.set_value(X_REF, x_ref_horizon)
    opti.set_value(Y_REF, y_ref_horizon)
    opti.set_value(V_REF, v_ref_horizon)
    opti.set_value(U_PREV, u_executed_prev)  # Push historical tracking input directly to solver memory

    # Seed guesses
    opti.set_initial(X, last_sol_X)
    opti.set_initial(U, last_sol_U)

    try:
        sol = opti.solve()
        u_control = sol.value(U[:, 0])
        last_sol_X = sol.value(X)
        last_sol_U = sol.value(U)
    except Exception as e:
        # Fallback sub-optimal state capture strategy
        u_control = opti.debug.value(U[:, 0])
        last_sol_X = opti.debug.value(X)
        last_sol_U = opti.debug.value(U)

    # Save the selected control inputs to match with the next tracking frame interval
    u_executed_prev = u_control.copy()

    # --- Plant Simulator (Process Step using Forward Euler) ---
    beta_sim = np.arctan(0.5 * np.tan(u_control[1]))
    x_next = x_current[0] + dt * (x_current[2] * np.cos(x_current[3] + beta_sim))
    y_next = x_current[1] + dt * (x_current[2] * np.sin(x_current[3] + beta_sim))
    v_next = x_current[2] + dt * u_control[0]
    psi_next = x_current[3] + dt * ((x_current[2] / l) * np.sin(beta_sim))

    is_turning_sharp = (u_control[1] >= np.deg2rad(3)) or (u_control[1] <= -np.deg2rad(3))
    turning.append(is_turning_sharp)
    
    x_current = np.array([x_next, y_next, v_next, psi_next])
    current_time += dt

    t_history.append(current_time)
    x_history.append(x_current[0])
    y_history.append(x_current[1])
    v_history.append(x_current[2])
    v_k_history.append(last_sol_X[2])
    v_ref_history.append(v_ref_horizon[0])
    acc_history.append(u_control[0])
    psi_history.append(x_current[3])
    delta_history.append(u_control[1])
    v_horizon.append(v_ref_horizon)

    if s_total_traveled >= track_length * track_percentual and x_current[2] < 0.01:
        print(f"Simulation completed! Lap finished and vehicle stopped on centerline at step {step}.")
        break

# Convert historical data into numpy arrays for rendering 
t_history = np.array(t_history)
x_history = np.array(x_history)
y_history = np.array(y_history)
v_history = np.array(v_history)
v_ref_history = np.array(v_ref_history)
acc_history = np.array(acc_history)
delta_history = np.array(delta_history)
delta_history = np.rad2deg(delta_history)

psi_history = np.array(psi_history)
psi_history = np.rad2deg(psi_history)
turning = [0 if t==False else 1 for t in turning]
turning = np.array(turning)

# %%
PlotSeriesPLY(xSeries=[t_history,t_history],ySeries=[v_ref_history,v_history],w=800,h=300)
PlotSeriesPLY(xSeries=[t_history],ySeries=[acc_history],w=800,h=300)

# %%
print(V_max)
print(V_cruising*1)

# %% [markdown]
# 4. PLOT RESULTS

# %%
data = [x_mid, y_mid, x_history,y_history, t_history, delta_history, psi_history, v_history]
PlotMPCTracksPLY(data,width=800,height=500)

step: 0
step: 250
step: 500
step: 750
step: 1000
step: 1250
step: 1500
step: 1750
step: 2000
step: 2250
step: 2500
step: 2750


25
16


In [447]:
V_turning

10